# Tema Programare: Transformarea Datelor Numerice

Bun venit la tema de programare despre Transformarea Datelor Numerice!

Datele numerice ajung rareori gata de folosit. Trasaturile din lumea reala acopera scale foarte diferite, urmeaza distributii asimetrice si combina date continue cu date bazate pe numarari. Stapanirea transformarii potrivite pentru situatia potrivita este una dintre cele mai valoroase abilitati de preprocesare - ea determina direct ce tipare pot detecta modelele tale.

**Ce vei face in aceasta tema:**

* **Comparatie intre metode de scalare** - Aplici StandardScaler, MinMaxScaler si RobustScaler; compari iesirile lor pe un set de date cu outlieri
* **Transformari de distributie** - Aplici transformarile log, square root, Box-Cox si Yeo-Johnson si masori reducerea asimetriei
* **Binning si discretizare** - Discretizezi trasaturi continue folosind bin-uri de latime egala, frecventa egala si bin-uri personalizate bazate pe cunostinte de domeniu
* **Impactul scalarii asupra KNN** - Masori si compari acuratetea KNN pe date brute vs. date scalate pentru a observa cum beneficiaza modelele bazate pe distanta de scalare
* **Pipeline de transformare** - Construiesti un `ColumnTransformer + Pipeline` complet care gestioneaza trasaturi numerice asimetrice si regulate
* **StandardScaler de la zero** - Implementezi standardizarea folosind doar NumPy (media, deviatia standard si formula scorului z)
* **MinMaxScaler de la zero** - Implementezi normalizarea min-max folosind doar NumPy (minimul, intervalul si formula de normalizare)

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI:</h4>

* Toate celulele sunt blocate, cu exceptia celor in care trebuie sa trimiti solutiile sau cand este mentionat explicit ca poti interactiona cu ele.

* In fiecare celula de exercitiu cauta comentariile `### ÎNCEPUT DE COD AICI ###` si `### SFÂRȘIT DE COD AICI ###`. Acestea iti arata unde sa scrii codul. **Nu adauga si nu modifica cod in afara acestor comentarii**.

* Poti adauga celule noi pentru experimente, dar acestea vor fi ignorate de evaluator, asa ca nu te baza pe celulele nou create pentru codul solutiei. Foloseste locurile oferite in notebook.

* Evita folosirea variabilelor globale daca nu este absolut necesar. Evaluatorul iti testeaza codul intr-un mediu izolat, fara sa ruleze toate celulele de sus in jos. Din acest motiv, variabilele globale pot sa nu fie disponibile la evaluare. Variabilele globale care trebuie folosite vor fi definite cu MAJUSCULE.

* Pentru a trimite notebook-ul la evaluare, salveaza-l mai intai apasand pe iconita 💾 din coltul stanga sus al paginii, apoi apasa pe butonul `Submit assignment` din coltul dreapta sus.
---


## Cuprins

- [Exercitiul 1 - Comparatie intre metode de scalare](#exercise-1)
- [Exercitiul 2 - Transformari de distributie](#exercise-2)
- [Exercitiul 3 - Binning si discretizare](#exercise-3)
- [Exercitiul 4 - Impactul scalarii trasaturilor asupra KNN](#exercise-4)
- [Exercitiul 5 - Pipeline de transformare](#exercise-5)
- [Exercitiul 6 - StandardScaler de la zero](#exercise-6)
- [Exercitiul 7 - MinMaxScaler de la zero](#exercise-7)


## Importuri si configurare


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests
from scipy import stats
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    FunctionTransformer,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## Set de date

Folosim un set de date sintetic de e-commerce cu 500 de clienti:

| Coloana | Distributie | Descriere |
|:-------|:-------------|:------------|
| `income` | Log-normala (asimetrie spre dreapta) | Venit anual |
| `age` | Uniforma | Varsta clientului |
| `price` | Normala cu outlieri | Pretul ultimei achizitii |
| `review_count` | Exponentiala | Numarul de recenzii scrise |
| `premium` | Tinta binara | 1 daca este abonat premium |


In [ ]:
np.random.seed(42)
N = 500

income       = np.random.lognormal(mean=10, sigma=1.2, size=N).round(2)
age          = np.random.randint(18, 70, N).astype(float)
price        = np.abs(np.random.normal(200, 80, N)).round(2)
review_count = (np.random.exponential(scale=10, size=N) + 1).astype(int).astype(float)

# Inject 5 outliers into price
price_w_outliers = price.copy()
price_w_outliers[:5] = [2000.0, 2500.0, 1800.0, 3000.0, 2200.0]

df = pd.DataFrame({'income': income, 'age': age,
                   'price': price_w_outliers, 'review_count': review_count})

logit = -2 + 0.5*(np.log(income)-10) + 0.02*age + np.random.normal(0, 1, N)
df['premium'] = (logit > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    df.drop('premium', axis=1), df['premium'], test_size=0.2, random_state=42)

print(f'Dataset shape: {df.shape}')
print(f'Premium rate: {df["premium"].mean():.2f}')
df.describe().round(2)

<a id='exercise-1'></a>
## Exercitiul 1 - Comparatie intre metode de scalare

### Context

Scaleri diferiti se potrivesc unor caracteristici diferite ale datelor:

| Scaler | Formula | Cel mai potrivit pentru |
|:-------|:--------|:---------|
| **StandardScaler** | $z = (x - \mu) / \sigma$ | Aproximativ Gaussian, fara outlieri puternici |
| **MinMaxScaler** | $x' = (x - \min) / (\max - \min)$ | Trasaturi limitate, retele neuronale |
| **RobustScaler** | $x' = (x - \text{median}) / \text{IQR}$ | Date cu outlieri reali |

> Fa intotdeauna fit doar pe **datele de antrenare**, apoi transforma atat setul de train, cat si pe cel de test.

### Sarcina

Implementeaza `apply_scalers(X_train, X_test, columns)` care:
1. Aplica **StandardScaler**, **MinMaxScaler** si **RobustScaler** pe `columns` specificate
2. Fiecare scaler face **fit doar pe `X_train`**, apoi transforma atat `X_train`, cat si `X_test`
3. Returneaza un dictionar: `{'standard': (X_tr, X_te), 'minmax': (X_tr, X_te), 'robust': (X_tr, X_te)}`
4. DataFrame-urile returnate contin valorile scalate ale coloanelor; celelalte coloane raman neschimbate

**Indicii:**
- `scaler.fit_transform(X_train[columns])` face fit si transformare intr-un singur apel.
- Foloseste `scaler.transform(X_test[columns])` (fara re-fit!) pentru datele de test.
- Atribuie inapoi array-ul scalat cu `result[columns] = scaled_array`.


In [ ]:
def apply_scalers(X_train, X_test, columns):
    ### ÎNCEPUT DE COD AICI ### (≈ 8 lines)
    results = {}
    for name, ScalerCls in [('standard', StandardScaler),
                             ('minmax',   MinMaxScaler),
                             ('robust',   RobustScaler)]:
        scaler = None
        X_tr_s = None   # fit on training, transform training
        X_te_s = None   # transform test (no re-fit)
        results[name] = (X_tr_s, X_te_s)
    ### SFÂRȘIT DE COD AICI ###

In [ ]:
NUMERIC_COLS = ['income', 'age', 'price', 'review_count']
scaled_results = apply_scalers(X_train, X_test, NUMERIC_COLS)

for name, (X_tr, _) in scaled_results.items():
    print(f'{name:12s} | mean: {X_tr[NUMERIC_COLS].mean().round(3).tolist()} '
          f'| std: {X_tr[NUMERIC_COLS].std().round(3).tolist()}')

In [ ]:
unittests.exercise_1(apply_scalers)

<a id='exercise-2'></a>
## Exercitiul 2 - Transformari de distributie

### Context

Distributiile asimetrice pot afecta modelele liniare si algoritmii sensibili la normalitate. Patru transformari frecvente:

| Transformare | Formula | Cerinta pentru intrare |
|:----------|:--------|:------------------|
| Log | $\ln(x)$ | Strict pozitiva |
| Square root | $\sqrt{x}$ | Nenegativa |
| Box-Cox | Putere optima $\lambda$ | Strict pozitiva |
| Yeo-Johnson | Putere optima $\lambda$ | Orice valoare reala |

### Sarcina

Implementeaza `apply_distribution_transforms(series)` care:
1. Aplica toate cele patru transformari pe Series-ul de intrare
2. Returneaza un dictionar `{'log': array, 'sqrt': array, 'boxcox': array, 'yeojohnson': array}`
   unde fiecare valoare este un array NumPy cu aceeasi lungime ca intrarea

**Indicii:**
- `np.log(series)` pentru transformarea log.
- `np.sqrt(series)` pentru square root.
- `stats.boxcox(series)` returneaza `(transformed_array, lambda_value)` - ia elementul 0.
- `stats.yeojohnson(series)` returneaza `(transformed_array, lambda_value)` - ia elementul 0.


In [ ]:
def apply_distribution_transforms(series):
    ### ÎNCEPUT DE COD AICI ### (≈ 6 lines)
    arr = series.values
    log_t   = None                    # np.log1p
    sqrt_t  = None                    # np.sqrt
    bc_t, _ = None                    # stats.boxcox  (returns array, lambda)
    yj_t, _ = None                    # stats.yeojohnson
    return {'log': log_t, 'sqrt': sqrt_t, 'boxcox': bc_t, 'yeojohnson': yj_t}
    ### SFÂRȘIT DE COD AICI ###

In [ ]:
transforms = apply_distribution_transforms(df['income'])

fig, axes = plt.subplots(1, 5, figsize=(18, 3))
all_data = [('Original', df['income'].values, 'Income')] + \
           [(k.capitalize(), v, k) for k, v in transforms.items()]
for ax, (title, data, _) in zip(axes, all_data):
    ax.hist(data, bins=30, alpha=0.8, edgecolor='white')
    ax.set_title(f'{title}\nSkew: {stats.skew(data):.2f}', fontsize=10)
    ax.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
unittests.exercise_2(apply_distribution_transforms)

<a id='exercise-3'></a>
## Exercitiul 3 - Binning si discretizare

### Context

Binning-ul transforma o trasatura continua in etichete intregi discrete:

| Strategie | Metoda | Cand se foloseste |
|:---------|:-------|:------------|
| Latime egala | `pd.cut` | Scala uniforma; intervale fizice cunoscute |
| Frecventa egala | `pd.qcut` | Date asimetrice; reprezentare egala a datelor |
| Margini personalizate | `pd.cut` cu margini explicite | Praguri bazate pe cunostinte de domeniu |

### Sarcina

Implementeaza `apply_binning(series, strategy, n_bins, custom_edges=None)` care:
1. Aplica strategia aleasa: `'equal_width'`, `'equal_freq'` sau `'custom'`
2. Pentru `'equal_width'`: foloseste `pd.cut(series, bins=n_bins, labels=False, include_lowest=True)`
3. Pentru `'equal_freq'`: foloseste `pd.qcut(series, q=n_bins, labels=False, duplicates='drop')`
4. Pentru `'custom'`: foloseste `pd.cut(series, bins=custom_edges, labels=False, include_lowest=True)`
5. Returneaza un `pd.Series` de tip integer cu indicii bin-urilor (bazati pe 0)

**Indicii:**
- `.astype(int)` pentru a te asigura ca iesirea este intreaga dupa `pd.cut` / `pd.qcut`.


In [ ]:
def apply_binning(series, strategy='equal_width', n_bins=5, custom_edges=None):
    ### ÎNCEPUT DE COD AICI ### (≈ 6 lines)
    if strategy == 'equal_width':
        binned = None     # pd.cut
    elif strategy == 'equal_freq':
        binned = None     # pd.qcut
    else:                 # 'custom'
        binned = None     # pd.cut with custom_edges
    return binned
    ### SFÂRȘIT DE COD AICI ###

In [ ]:
income_ew = apply_binning(df['income'], strategy='equal_width',   n_bins=5)
income_ef = apply_binning(df['income'], strategy='equal_freq',    n_bins=5)
age_custom = apply_binning(df['age'], strategy='custom',
                            custom_edges=[0, 25, 35, 50, 100])

print('Equal-width bin counts:',     income_ew.value_counts().sort_index().tolist())
print('Equal-freq bin counts:',      income_ef.value_counts().sort_index().tolist())
print('Custom age-group bin counts:', age_custom.value_counts().sort_index().tolist())

In [ ]:
unittests.exercise_3(apply_binning)

<a id='exercise-4'></a>
## Exercitiul 4 - Impactul scalarii trasaturilor asupra KNN

### Context

KNN calculeaza **distante Euclidiene** intre puncte. Cand trasaturile au scale foarte diferite, trasatura cu scala mare domina calculul distantei, iar modelul o ignora efectiv pe cealalta - chiar daca aceea contine semnalul real.

Dupa `StandardScaler`, fiecare trasatura contribuie in mod egal la distanta, permitand KNN sa foloseasca toate trasaturile.

### Sarcina

Implementeaza `compare_knn_scaling(X_train, y_train, X_test, y_test, k=5)` care:
1. Antreneaza un `KNeighborsClassifier(n_neighbors=k)` pe trasaturile **brute** si evalueaza acuratetea
2. Aplica `StandardScaler` (fit pe `X_train`, transformare pe ambele seturi) si antreneaza un al doilea KNN
3. Returneaza `{'unscaled_accuracy': float, 'scaled_accuracy': float}`

**Indicii:**
- Foloseste `accuracy_score(y_test, model.predict(X_test))` pentru evaluare.
- `X_train` / `X_test` sunt array-uri NumPy, nu DataFrame-uri.


In [ ]:
def compare_knn_scaling(X_train, y_train, X_test, y_test, k=5):
    ### ÎNCEPUT DE COD AICI ### (≈ 8 lines)
    # unscaled
    knn_raw = None
    unscaled_acc = None
    # scaled
    scaler = None
    X_tr_s = None
    X_te_s = None
    knn_sc  = None
    scaled_acc = None
    return {'unscaled_accuracy': unscaled_acc, 'scaled_accuracy': scaled_acc}
    ### SFÂRȘIT DE COD AICI ###

In [ ]:
# Build a dataset where one feature is signal and the other is high-scale noise
np.random.seed(42)
n_demo = 300
X_demo = np.column_stack([
    np.concatenate([np.random.randn(n_demo//2),       # class 0 feature 1
                    np.random.randn(n_demo//2) + 5]), # class 1 feature 1 (signal)
    np.concatenate([np.random.randn(n_demo//2)*800,   # class 0 feature 2 (noise)
                    np.random.randn(n_demo//2)*800]),  # class 1 feature 2 (noise)
])
y_demo = np.array([0]*(n_demo//2) + [1]*(n_demo//2))
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(X_demo, y_demo,
                                                    test_size=0.3, random_state=42)

result = compare_knn_scaling(X_tr_d, y_tr_d, X_te_d, y_te_d)
print(f'Unscaled KNN accuracy: {result["unscaled_accuracy"]:.3f}')
print(f'Scaled   KNN accuracy: {result["scaled_accuracy"]:.3f}')

In [ ]:
unittests.exercise_4(compare_knn_scaling)

<a id='exercise-5'></a>
## Exercitiul 5 - Pipeline de transformare

### Context

Un pipeline de transformare numerica pregatit pentru productie aplica transformari diferite pe coloane diferite in interiorul unui `ColumnTransformer`:

```
Input DataFrame
  ├─ log_cols  ──► log1p()  ──► StandardScaler ──┐
  └─ other_cols ──────────► StandardScaler ───────┼──► Estimator
```

Folosirea `log1p` (in loc de `log`) gestioneaza in siguranta valorile zero. Impachetarea tuturor pasilor intr-un `Pipeline` previne scurgerea de informatie in timpul cross-validarii.

### Sarcina

Implementeaza `create_transformation_pipeline(numeric_cols, log_cols)` care:
1. Pentru `log_cols`: creeaza un sub-`Pipeline` cu `FunctionTransformer(np.log1p)` apoi `StandardScaler`
2. Pentru coloanele din `numeric_cols` dar **nu** din `log_cols`: aplica direct `StandardScaler`
3. Le combina pe toate prin `ColumnTransformer`
4. Returneaza un `Pipeline([('preprocessing', ct), ('clf', LogisticRegression(max_iter=500))])`

**Indicii:**
- `[c for c in numeric_cols if c not in log_cols]` iti da coloanele non-log.
- `FunctionTransformer(np.log1p)` impacheteaza `np.log1p` ca transformator sklearn.


In [ ]:
def create_transformation_pipeline(numeric_cols, log_cols):
    ### ÎNCEPUT DE COD AICI ### (≈ 7 lines)
    other_cols = None       # columns in numeric_cols but not in log_cols
    ct = ColumnTransformer([
        ('log',    None, log_cols),    # FunctionTransformer + StandardScaler
        ('std',    None, other_cols),  # StandardScaler only
    ])
    return Pipeline([('preprocessing', ct), ('clf', LogisticRegression(max_iter=1000))])
    ### SFÂRȘIT DE COD AICI ###

In [ ]:
from sklearn.model_selection import cross_val_score

LOG_COLS = ['income', 'review_count']
ALL_COLS = ['income', 'age', 'price', 'review_count']

pipeline = create_transformation_pipeline(ALL_COLS, LOG_COLS)
X_pipe = df[ALL_COLS]
y_pipe = df['premium']

scores = cross_val_score(pipeline, X_pipe, y_pipe, cv=5, scoring='accuracy')
print(f'5-fold CV accuracy: {scores.mean():.3f} ± {scores.std():.3f}')

In [ ]:
unittests.exercise_5(create_transformation_pipeline)

<a id='exercise-6'></a>
## Exercitiul 6 - StandardScaler de la zero

### Context

Normalizarea z-score centreaza si scaleaza fiecare trasatura:

$$z = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$

Regula-cheie: **$\mu$ si $\sigma$ sunt calculate doar din setul de antrenare**, apoi sunt aplicate atat pe train, cat si pe test.

### Sarcina

Implementeaza `standard_scaler_from_scratch(X_train, X_test)` unde `X_train` si `X_test` sunt array-uri NumPy 2D. Functia trebuie:
1. Sa calculeze **media pe coloane** si **deviatia standard de populatie** (`ddof=0`) din `X_train`
2. Sa scaleze `X_train` si `X_test` folosind aceste statistici
3. Sa returneze `(X_train_scaled, X_test_scaled)` ca `np.ndarray`

**Indicii:**
- `X_train.mean(axis=0)` calculeaza mediile pe coloane.
- `X_train.std(axis=0)` foloseste implicit `ddof=0` (deviatia standard de populatie) - in concordanta cu sklearn.


In [ ]:
def standard_scaler_from_scratch(X_train, X_test):
    ### ÎNCEPUT DE COD AICI ### (≈ 5 lines)
    mean = None     # compute column-wise mean from X_train
    std  = None     # compute column-wise population std from X_train
    X_train_scaled = None   # apply z-score formula
    X_test_scaled  = None   # use training mean/std
    return X_train_scaled, X_test_scaled
    ### SFÂRȘIT DE COD AICI ###

In [ ]:
X_np = df[['income', 'age', 'price', 'review_count']].values
X_tr_np = X_np[:400]
X_te_np = X_np[400:]

X_tr_sc, X_te_sc = standard_scaler_from_scratch(X_tr_np, X_te_np)
print(f'Train mean  (should be ≈ 0): {X_tr_sc.mean(axis=0).round(6).tolist()}')
print(f'Train std   (should be ≈ 1): {X_tr_sc.std(axis=0).round(6).tolist()}')
print(f'Test mean   (info only):      {X_te_sc.mean(axis=0).round(3).tolist()}')

In [ ]:
unittests.exercise_6(standard_scaler_from_scratch)

<a id='exercise-7'></a>
## Exercitiul 7 - MinMaxScaler de la zero

### Context

Normalizarea Min-Max mapeaza fiecare trasatura in intervalul $[0, 1]$:

$$x' = \frac{x - x_{\min,\text{train}}}{x_{\max,\text{train}} - x_{\min,\text{train}}}$$

Din nou, minimul si maximul sunt calculate **doar din setul de antrenare**. Valorile de test pot cadea in afara intervalului $[0, 1]$ - acest lucru este asteptat si corect.

### Sarcina

Implementeaza `minmax_scaler_from_scratch(X_train, X_test)` unde intrarile sunt array-uri NumPy 2D. Functia trebuie:
1. Sa calculeze **minimul si maximul pe coloane** din `X_train`
2. Sa aplice formula pentru a scala atat `X_train`, cat si `X_test`
3. Sa returneze `(X_train_scaled, X_test_scaled)` ca `np.ndarray`

**Indicii:**
- `X_train.min(axis=0)` si `X_train.max(axis=0)` pentru min/max pe coloane.
- Calculeaza o singura data `X_range = X_max - X_min` si refoloseste-l.


In [ ]:
def minmax_scaler_from_scratch(X_train, X_test):
    ### ÎNCEPUT DE COD AICI ### (≈ 5 lines)
    X_min   = None   # column-wise min from X_train
    X_range = None   # column-wise (max - min) from X_train
    X_train_scaled = None   # apply min-max formula
    X_test_scaled  = None   # use training min/range
    return X_train_scaled, X_test_scaled
    ### SFÂRȘIT DE COD AICI ###

In [ ]:
X_tr_mm, X_te_mm = minmax_scaler_from_scratch(X_tr_np, X_te_np)
print(f'Train min   (should be ≈ 0): {X_tr_mm.min(axis=0).round(6).tolist()}')
print(f'Train max   (should be ≈ 1): {X_tr_mm.max(axis=0).round(6).tolist()}')
print(f'Test  range (may exceed [0,1]): min={X_te_mm.min(axis=0).round(3).tolist()}, '
      f'max={X_te_mm.max(axis=0).round(3).tolist()}')

In [ ]:
unittests.exercise_7(minmax_scaler_from_scratch)